# 04. Fluctuations and Transient Events

An equilibrium is a time average. Everything interesting about a discharge is the part that
would not hold still long enough to be averaged.

Session 03 handed over an equilibrium you can interrogate rather than assume. This session
reads what oscillates on top of it — and spends as much effort on a harder question: how much
of that reading are you actually allowed to claim?

## Session Overview

By the end of this session you will be able to:

- place a discharge's events on one axis using the timing helpers rather than by eye;
- read a Mirnov spectrogram and say which features belong to the plasma and which to the transform;
- fit a spectral index and say what it is an index *of*;
- measure a transient as a change in band power against a floor you also measured;
- fit a toroidal mode number and state the alias step that comes with it;
- locate the rational surfaces an equilibrium actually has, and say where the chain from a mode
  to a surface stops.

Two shots, because no single packaged VEST discharge can do both halves:

| shot | what it brings |
| --- | --- |
| 39915 | 64 fast probes at 250 kHz and nine equilibrium slices — and a toroidal array that is described but not acquired |
| 45531 | the outboard fluctuation array: ten probes at each of 45, 135 and 225 degrees |

Everything on 39915 runs from the installed package. The 45531 record lives in the repository,
and the cell that loads it says so.

## Physical Context

### What a pickup coil measures

A Mirnov coil measures the flux change through its winding, so its output is a *time
derivative*:

$$V(t) \propto \frac{\mathrm{d}B}{\mathrm{d}t}
\qquad\Longrightarrow\qquad
S_{\mathrm{d}B/\mathrm{d}t}(f) = (2\pi f)^2\, S_B(f).$$

A spectral index fitted to raw pickup voltage is therefore the field's index **plus two**. VAFT
does not integrate silently behind an axis label: the spectrum axis will say `V^2/Hz` and mean it.

### What an array can resolve

A coherent perturbation read around the torus has a phase that falls on a line,

$$\varphi_{\text{phase}} = \varphi_0 - n\,\phi .$$

Phases are read modulo a turn. If the probes that recorded sit at angles all spaced by
$\Delta\phi$, then every $n$ differing by $360^\circ/\Delta\phi$ predicts the *same* phase at
every probe, and no fit can separate them. Two positions 120 degrees apart give $n$ modulo 3;
three positions 90 degrees apart give $n$ modulo 4. State the modulus or you have not stated
the result.

### Transients, and what VAFT will not call them

VAFT has no IRE, sawtooth, ELM or disruption classifier. What it has is generic: onset
primitives such as `active_window` — whose collapse fallback ends a window at the last steep
fall when a signal never returns to baseline, which is what a terminating discharge does —
together with `robust_peak` and `sustained_excess_onset`, and the timing wrappers built on them.

So this session says *onset* and *termination*, and names nothing it cannot support.

## Load / Prepare Data

### The packaged discharge

Shot 39915 again, the discharge sessions 02 and 03 built up. No arguments, no network.

In [ ]:
import numpy as np
import vaft
import matplotlib.pyplot as plt

In [ ]:
ods = vaft.omas.sample_ods()

# Channel indices depend on which archive fields existed for a shot, so every
# selection below goes through a name rather than a position.
probe_names = [
    str(ods[f"magnetics.b_field_pol_probe.{index}.name"])
    for index in range(len(ods["magnetics.b_field_pol_probe"]))
]
recorded = [
    index for index in range(len(probe_names))
    if "voltage" in ods[f"magnetics.b_field_pol_probe.{index}"]
]
print(f"probes described: {len(probe_names)} | probes that recorded: {len(recorded)}")

### When was there a plasma, and what fired first

Session 02 found a breakdown time. Here the same question is put to the two wrappers that carry
their own provenance, because every window in this session is cut with them rather than chosen
by eye.

In [ ]:
from vaft.omas.plasma_timing import plasma_timing
from vaft.omas.discharge_timing import discharge_timing

timing = plasma_timing(ods)
events = discharge_timing(ods)

print(f"plasma window : {timing.onset * 1e3:.2f} - {timing.offset * 1e3:.2f} ms")
print(f"window source : {timing.source}")
print(f"agreement     : {timing.agreement}")
print(f"OH onset      : {events.oh_onset * 1e3:.2f} ms  ({events.oh_coil})")
print(f"V_loop zero   : {events.vloop_time * 1e3:.2f} ms")

### What this input supports

Session 02 used `available_plots` to find out what an ODS can draw. Use it again here, and keep
in mind what the answer means: availability reports whether a call will *raise*, not whether its
answer will be worth having. This session comes back to that distinction.

In [ ]:
names = {row["name"] for row in vaft.omas.available_plots(ods)}
for plot in ("mirnov_time_voltage", "mirnov_spectrogram",
             "mirnov_spectrum", "mirnov_spatial_phase"):
    print(f"{plot:22s} {plot in names}")

field_validity = {
    int(ods[f"magnetics.b_field_pol_probe.{index}.field.validity"])
    for index in range(len(probe_names))
    if "field" in ods[f"magnetics.b_field_pol_probe.{index}"]
}
print(f"field.validity values present: {sorted(field_validity)}")

All four Mirnov views are available. The integrated field, though, is flagged `-2` on every
channel that carries it — invalid, not merely suspect — so this session works from `voltage`
throughout. That is a stated reason, not a preference: a plot drawn from the field here would
be drawn from data the shot itself disowns.

## Guided Analysis

### Three probes, three places

An inboard probe, an outboard-midplane probe and one low on the vessel. The outboard one carries
the rest of the session.

In [ ]:
selected = [
    probe_names.index(name)
    for name in ("MagneticFieldProbe_H2-05_Bz",
                 "MagneticFieldProbe_C2-05_Bz",
                 "MagneticFieldProbe_L-04_Bz")
]
vaft.omas.plot_mirnov_time_voltage(
    ods, channels=selected, x_limits=(0.300, 0.336), layout="subplots"
)
plt.show()

What you are looking at is already a derivative, so the envelope is not the field's amplitude —
the axis says `V` and means it. Note also that `layout=` is a real choice rather than a style
one: `overlay` to compare traces against each other, `subplots` to read each on its own scale.

In [ ]:
figure, axes = vaft.omas.plot_mirnov_spectrogram(
    ods, channel=selected[1], method="stft",
    time_range=(0.290, 0.340), nperseg=1024, max_frequency=6.0e4,
)
axes.axvline(timing.onset, color="w", ls="--", lw=0.9)
axes.axvline(timing.offset, color="w", ls="--", lw=0.9)
plt.show()

### Two transients, named carefully

Read it left to right. Nothing structured before the first dashed line; a band appearing a few
milliseconds after the onset and drifting downward as the discharge proceeds; a broadband burst
at the offset.

Those are an onset and a termination. They are not called a disruption, because nothing in VAFT
decided that. And the drift is not attributed to anything, because separating a mode's own
frequency from the plasma's rotation needs a rotation measurement this shot does not carry.

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
vaft.omas.plot_mirnov_spectrogram(
    ods, channel=selected[1], method="stft",
    time_range=(0.306, 0.332), nperseg=512, max_frequency=4.0e4, ax=axes[0],
)
vaft.omas.plot_mirnov_spectrogram(
    ods, channel=selected[1], method="hann_fft",
    time_range=(0.306, 0.332), window_size=500, max_frequency=4.0e4, ax=axes[1],
)
axes[0].set_title('method="stft"')
axes[1].set_title('method="hann_fft"')
axes[0].set_xlabel("")
plt.show()

### One signal, two transforms

`method=` selects scipy's short-time Fourier transform or VAFT's own Hann-window FFT, which is
kept because the VEST Mirnov analyses were written against it. Both are given a window of about
the same length on purpose, so that switching method changes the algorithm and not, silently,
the resolution as well.

The trade is the same either way: a window of duration $T$ resolves $\Delta f \approx 1/T$ and
blurs anything faster than $T$. A feature that survives both transforms is in the signal. A
feature that appears in only one is in the transform.

In [ ]:
from vaft.process.fluctuation import compute_psd, fit_power_law_spectrum

vaft.omas.plot_mirnov_spectrum(
    ods, channel=selected[1], time_range=(0.316, 0.326), nperseg=1024,
    fit_ranges=[(2.0e3, 1.5e4), (1.5e4, 1.2e5)],
    series_label="C2-05 voltage",
)
plt.show()

base = f"magnetics.b_field_pol_probe.{selected[1]}.voltage"
time_axis = np.asarray(ods[f"{base}.time"]).ravel()
voltage = np.asarray(ods[f"{base}.data"]).ravel()
inside = (time_axis >= 0.316) & (time_axis <= 0.326)
spectrum = compute_psd(time_axis[inside], voltage[inside], nperseg=1024)

for low, high in ((2.0e3, 1.5e4), (1.5e4, 1.2e5)):
    fit = fit_power_law_spectrum(spectrum.frequency, spectrum.psd, f_range=(low, high))
    print(f"{low / 1e3:5.0f}-{high / 1e3:5.0f} kHz : alpha(dB/dt) = {fit.alpha:+.2f}  "
          f"R^2 = {fit.r_squared:.3f}  ->  alpha(B) = {fit.alpha - 2.0:+.2f}")

Three readings, in order.

The y axis says `V^2/Hz`, so both indices are `dB/dt`'s. The `alpha(B)` column is the subtraction
from the Physical Context and nothing more — no integration happened.

$R^2$ is part of the result. A fit spanning a band that contains a coherent peak reports a poor
one, and that is the fit telling you the model is wrong *there*. The response is to move the
band, not to quote the slope anyway.

And VAFT supplies no reference slopes, ever. `reference_slopes=` draws yours, with your label,
and attaches no meaning to the number.

**For the depth**, `notebooks/fluctuation_diagnostics_analysis.ipynb` takes this considerably
further on shot 45531: Welch's resolution-against-variance trade, linear-phase against IIR
filtering, an imposed against a searched spectral break, the `+2` shift measured rather than
asserted, and the whole analysis closed against a second diagnostic. This session teaches the
reading; that notebook is the depth.

In [ ]:
from vaft.process.fluctuation import compute_spectrogram, compute_band_power

evolution = compute_spectrogram(time_axis, voltage, window_duration=1.0e-3, overlap=0.75)
band_power = np.array([
    compute_band_power(evolution.frequency, column, {"mhd": (2.0e3, 4.0e4)})["mhd"]
    for column in evolution.magnitude.T
])

figure, axes = plt.subplots()
axes.semilogy(evolution.time, band_power)
axes.axvspan(timing.onset, timing.offset, alpha=0.15, label="plasma window")
axes.axvline(events.oh_onset, color="k", ls="--", lw=0.8, label="OH onset")
axes.set_xlabel("Time [s]")
axes.set_ylabel("2-40 kHz band power [V$^2$]")
axes.legend()
axes.grid(alpha=0.3)
plt.show()

quiet = np.median(band_power[evolution.time < events.oh_onset])
active = np.median(
    band_power[(evolution.time >= timing.onset) & (evolution.time <= timing.offset)]
)
print(f"band power before the coils fired  : {quiet:.2e}")
print(f"band power inside the plasma window: {active:.2e}  ({active / quiet:.0f}x)")
print("fluctuation power rises above the pre-discharge floor:", bool(active > 10.0 * quiet))

### A transient is a change measured against a floor

The floor is not zero and it is not noise-free: before the coils fire the band already carries
pickup. Measuring the rise against *that* rather than against zero is the entire content of the
claim, and it is why the number printed above is a ratio.

Note what the figure does not do. It integrates a band a reader named, and `compute_band_power`
attaches no meaning to the name.

### The toroidal mode number

One call, and it is advertised on this input. Draw the measured points on their own first:
nothing appears on a VAFT plot that the caller did not ask for, so `show_fit=False` gives the
phases with no line through them.

In [ ]:
vaft.omas.plot_mirnov_spatial_phase(
    ods, time=0.3150, num_modes=1, candidate_n=range(0, 7), show_fit=False
)
plt.show()

In [ ]:
vaft.omas.plot_mirnov_spatial_phase(
    ods, time=0.3150, num_modes=1, candidate_n=range(0, 7)
)
plt.show()

Read the title before the number. It states how many *distinct toroidal positions* answered, and
the legend carries a modulus. Both are the plot telling you what it could not determine.

Now ask where those positions came from.

In [ ]:
angles = sorted({
    round(float(np.degrees(ods[f"magnetics.b_field_pol_probe.{index}.position.phi"])), 1)
    for index in recorded
})
spacing = np.diff(np.array(angles + [angles[0] + 360.0]))
print("toroidal angles that recorded:", angles)
print(f"smallest spacing {spacing.min():.0f} deg -> "
      f"n is resolved modulo {round(360.0 / spacing.min())}")

reference = probe_names.index("MagneticFieldProbe_C2-05_Bz")
twin = next(
    index for index in recorded
    if str(ods[f"magnetics.b_field_pol_probe.{index}.identifier"]).endswith(":phase_reference")
)
print(f"entry at phi =   0 deg: {ods[f'magnetics.b_field_pol_probe.{reference}.identifier']}")
print(f"entry at phi = 240 deg: {ods[f'magnetics.b_field_pol_probe.{twin}.identifier']}")
print("the two toroidal entries carry identical samples:", bool(np.array_equal(
    np.asarray(ods[f"magnetics.b_field_pol_probe.{reference}.voltage.data"]),
    np.asarray(ods[f"magnetics.b_field_pol_probe.{twin}.voltage.data"]),
)))

VEST describes four toroidal reference probes at 0, 120, 180 and 240 degrees, all at the same
$(R, Z)$. In this shot three of them recorded nothing. The fourth, `C2-05`, recorded — and it
appears twice: once in the equilibrium array, where every probe is declared at $\phi = 0$, and
once as a `:phase_reference` entry carrying its true 240 degrees. Same channel, same samples.

So a fit across those two positions compares a signal with itself, and the phase difference is
exactly zero by construction. The remaining probes all sit at one angle, where their phase
spread is poloidal structure being read as though it were toroidal.

**The packaged shot offers one physical toroidal position.** The plot did not lie: its title
said *2 toroidal positions*, and two is the documented minimum, not a claim of sufficiency.

This is the converse of the habit sessions 02 and 03 taught. There, a refused plot was often an
underived quantity, and the fix was to derive it. Here a plot is offered, runs, and produces a
number — and the number is an identity. **An offered plot is not automatically an answer worth
having**, and `available_plots` never promised that it was: it reports whether a call will raise.

The duplicate entry is a mapping defect rather than a property of the machine, and it is tracked
as issue [#724](https://github.com/VEST-Tokamak/vaft/issues/724).

## Interpretation Checkpoints

Each of these has a definite answer in what you have already plotted.

1. **The band drifts downward** while the current decays and `q95` falls. Name two different
   explanations for a falling frequency, and say what measurement would separate them. One of
   the two is not available on this shot — which?
2. **The phase plot shows many points at one toroidal angle**, spread over nearly a full turn.
   What is that spread, if it is not $n$?
3. **`available_plots` reported `mirnov_spatial_phase` as available.** Was that report wrong?
   What would an availability check have to know to answer the question you actually asked?
4. **The spectrum fit reports $R^2 = 0.54$ on one band and $0.83$ on the other.** Which of the
   two indices would you quote, and what would you quote it *as*?
5. **The band power rises by a factor of about a hundred.** Over what interval is that ratio
   defined, and what would happen to it if you had started the comparison after the coils fired?

## Integrated Analysis

### A shot whose array can answer

To fit a toroidal mode number you need probes that are not the same probe. Shot 45531 has them:
the outboard fluctuation array, ten probes at each of three toroidal positions. Their
identifiers carry the VEST clock angles 45, 135 and 225 degrees; the IMAS toroidal angles the
cells below print are the reflection of those, 315, 225 and 135, because the port clock runs
clockwise and $\phi$ does not. Either way the spacing is 90 degrees, which resolves $n$ to
within an alias step of 4 rather than 3 — and, more to the point, they are three different
instruments.

The raw archive record for 45531 lives in the repository rather than the installed package, so
this cell needs a source checkout. Nothing above this point does, and nothing below it changes
what you have already learned from 39915.

One more thing to expect: the mapper announces every channel the archive did not hold. Three of
the names it reports are the toroidal reference probes that were missing from 39915 as well.
That output is left in — it is the same fact, said out loud by the mapper rather than uncovered
by hand.

In [ ]:
from omas import ODS
from vaft.machine_mapping.magnetics import magnetics

template = str(vaft.data.data_path("legacy/shot_45531.json.gz").parent / "shot_{shot}.json.gz")
fluctuation = ODS()
magnetics(fluctuation, shot=45531, tstart=0.26, tend=0.36, dt=4.0e-5, raw_source=template)

fluctuation_names = [
    str(fluctuation[f"magnetics.b_field_pol_probe.{index}.name"])
    for index in range(len(fluctuation["magnetics.b_field_pol_probe"]))
]
outboard = [fluctuation_names.index(f"OutMirnov_{angle}_L1-03") for angle in (45, 135, 225)]
print("probes mapped:", len(fluctuation_names), "| the outboard trio:", outboard)

In [ ]:
fluctuation_timing = plasma_timing(fluctuation)
print(f"plasma window : {fluctuation_timing.onset * 1e3:.2f} - "
      f"{fluctuation_timing.offset * 1e3:.2f} ms")
print(f"window source : {fluctuation_timing.source}")
print(f"why not the light: {fluctuation_timing.fallback_reason}")

Only magnetics was mapped here, so the filterscope could not answer and the plasma current did.
The record names the check that failed rather than going quiet about it — the same helper, a
different source, and it says which.

In [ ]:
vaft.omas.plot_mirnov_time_voltage(
    fluctuation, channels=outboard, x_limits=(0.2960, 0.3080), layout="subplots"
)
plt.show()

In [ ]:
vaft.omas.plot_mirnov_spatial_phase(
    fluctuation, time=0.3025, num_modes=1, candidate_n=range(0, 7)
)
plt.show()

On the shot that *can* answer, the plain call gives the worse answer. The probes it left out are
exactly the ones that could resolve $n$: they are digitised faster than the majority, and phases
read off two different digitisations are not comparable sample by sample. One timebase per
figure is the rule, and the title states what it cost.

Choosing the channels is the analysis.

In [ ]:
vaft.omas.plot_mirnov_spatial_phase(
    fluctuation, time=0.3025, channels=outboard,
    num_modes=2, candidate_n=range(0, 5), window_size=2000,
)
plt.show()

outboard_angles = sorted({
    round(float(np.degrees(
        fluctuation[f"magnetics.b_field_pol_probe.{index}.position.phi"])), 1)
    for index in outboard
})
spacing = np.diff(np.array(outboard_angles + [outboard_angles[0] + 360.0]))
print("toroidal angles that recorded:", outboard_angles)
print(f"smallest spacing {spacing.min():.0f} deg -> "
      f"n is resolved modulo {round(360.0 / spacing.min())}")

A number, with its limit attached. Only one of the two bands carries a modulus in its label,
because only that one has an alias inside the candidate set it was given — the label appears
when the ambiguity is real, not as decoration.

Keep one caution in view: with three probes the fitted line has a single degree of freedom left,
so a small residual is weak evidence. Three points nearly always lie near a line.

### The two transients on one axis

Back to 39915, because 45531 was mapped with magnetics alone and has no equilibrium. That split
is itself the point: the shot that can measure the mode is not the shot that can locate its
surface, and no single packaged VEST discharge does both.

In [ ]:
figure, axes = plt.subplots(3, 1, sharex=True, figsize=(8, 8))

vaft.omas.plot_plasma_current_time(ods, ax=axes[0])
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha", ax=axes[1])
axes[2].semilogy(evolution.time, band_power)
axes[2].set_ylabel("2-40 kHz band power [V$^2$]")
axes[2].grid(alpha=0.3)

for axis in axes:
    axis.axvline(timing.onset, color="k", ls="--", lw=0.8)
    axis.axvline(timing.offset, color="k", ls="--", lw=0.8)
for axis in axes[:-1]:
    axis.set_xlabel("")
axes[2].set_xlabel("Time [s]")
plt.show()

The current, the light and the fluctuation power share one axis and one pair of dashed lines.
Note the sampling: the filterscope runs at 25 kHz, so its Nyquist frequency is 12.5 kHz — about
two and a half samples per cycle of the band in the spectrogram. It can corroborate the
*envelope* of the activity. It cannot corroborate the mode.

### Does the equilibrium have a surface for it?

A mode resonates where $q = m/n$. VAFT will locate the normalised radius of a bare $q$ value;
it has no resolver keyed on $(m, n)$. So ask the question VAFT can answer, and stop where it
cannot.

In [ ]:
from vaft.process.equilibrium import as_equilibrium, derive_global_descriptors

times = np.asarray(ods["equilibrium.time"])
usable = [
    index for index in range(times.size)
    if float(ods[f"equilibrium.time_slice.{index}.global_quantities.ip"]) != 0.0
]
tracked = (3.0, 4.0, 5.0)
surfaces = {q: [] for q in tracked}

header = f"{'t [ms]':>7} {'q0':>6} {'q95':>6} " + " ".join(
    f"{'q=' + str(int(q)):>7}" for q in tracked
)
print(header)
for index in usable:
    descriptors = derive_global_descriptors(
        as_equilibrium(ods, time_index=index), rational_q=(1.0, 2.0, 3.0, 4.0, 5.0)
    )
    row = []
    for q in tracked:
        found = descriptors.rational_surfaces[q]
        surfaces[q].append(float(found[0].value) if found else np.nan)
        row.append(f"{surfaces[q][-1]:7.3f}" if found else f"{'--':>7}")
    print(f"{times[index] * 1e3:7.1f} {descriptors['q0'].value:6.2f} "
          f"{descriptors['q95'].value:6.2f} " + " ".join(row))

In [ ]:
figure, axes = plt.subplots()
for q in tracked:
    axes.plot(times[usable] * 1e3, surfaces[q], marker="o", label=f"q = {q:.0f}")
axes.set_xlabel("Time [ms]")
axes.set_ylabel(r"$\psi_N$ of the surface")
axes.set_ylim(0.0, 1.0)
axes.legend()
axes.grid(alpha=0.3)
plt.show()

In [ ]:
descriptors = derive_global_descriptors(
    as_equilibrium(ods, time_index=usable[4]), rational_q=(1.0, 2.0, 3.0, 4.0, 5.0)
)
print(f"q on axis: {descriptors['q0'].value:.2f}")
for q in (1.0, 2.0, 3.0, 4.0, 5.0):
    print(f"q = {q:.0f} surface in this equilibrium: {bool(descriptors.rational_surfaces[q])}")

The closing argument, in four moves.

**What 39915 gives.** A band that two independent transforms agree on, carrying about a hundred
times the pre-discharge power, drifting downward as the current decays. Real, and unlabelled:
the shot's array offers one physical toroidal position, so it carries no toroidal mode number
at all.

**What 45531 gives.** Three probes, three genuine angles 90 degrees apart, and a fitted $n$ with
its alias step printed beside it. A number, with its limit attached.

**Where the chain stops.** Turning $n$ into a resonance needs $m$ and needs the surface. $q$ on
axis is above two in every slice of this equilibrium, so there is **no $q = 1$ and no $q = 2$
surface** — the familiar $m/n = 1/1$ sawtooth and $2/1$ tearing readings are unavailable as a
*fact about this equilibrium*, not as a missing feature. Where $q = 3$ does exist it sweeps
across a third of the minor radius in ten milliseconds, so "the $q = 3$ surface" is not even a
fixed place. And pairing it with a measured $n$ needs $m$, which no VEST diagnostic in VAFT
measures, plus a rotation frequency to separate the plasma frame from the lab frame, which none
measures either.

**What you may write down.** The frequency and its drift; the band power against a measured
floor; $n$ modulo the array's alias step; the normalised radius of each bare $q$ surface at each
slice; and the sentence that the pairing between them is undetermined. That last sentence is a
result, not a failure, and it is the one a reviewer will check first.

## Independent Exercise

Take the array further than this session did.

The outboard array has more than one row. Repeat the toroidal fit on the second row and decide
whether the two rows agree; then walk the fit across the discharge and say whether $n$ holds
while the frequency drifts. The answer to that second question is not stated here on purpose.

The last two steps are **lab mode**: they need an optional package and a second archive record.
Uncomment and run them if you have both; read them if you do not. Nothing above this point
requires either.

**Two things this session cannot compute**, named rather than approximated:

- **A mode's resonant surface from $(m, n)$.** VAFT gives $\psi_N$ for a bare $q$ and has no
  resolver keyed on a mode's own numbers — issue
  [#506](https://github.com/VEST-Tokamak/vaft/issues/506), open. Writing one means first
  deciding where $m$ would come from, and on VEST today the answer is nowhere.
- **A mode track on a spectrogram.** Drawing $f = n\,f_\phi$ over a time-frequency map needs a
  rotation frequency. Neither shot here has one — issue
  [#460](https://github.com/VEST-Tokamak/vaft/issues/460), open.

In [ ]:
# 1. The same three angles, one row further round: OutMirnov_*_L2-03.
# second_row = [fluctuation_names.index(f"OutMirnov_{angle}_L2-03") for angle in (45, 135, 225)]
# vaft.omas.plot_mirnov_spatial_phase(
#     fluctuation, time=0.3025, channels=second_row,
#     num_modes=2, candidate_n=range(0, 5), window_size=2000,
# )
# plt.show()
#
# 2. Do the two rows agree on n? If they do not, which do you believe, and what
#    would you have to look at to decide?
#
# 3. Walk the fit across the discharge -- 296, 300, 302.5, 305 ms -- and record
#    the frequency and the n at each. Does n hold while f drifts?
#
# 4. Not every probe in the array shares one digitiser rate. Mix a slower one in
#    and read what the title then says about the timebase it dropped.
#
# 5. A third transform, through the optional fcwt package. This is lab mode:
#    method="cwt" analyses one named band and refuses without frequency_range=.
# vaft.omas.plot_mirnov_spectrogram(
#     fluctuation, channel=outboard[0], method="cwt",
#     time_range=(0.294, 0.308), frequency_range=(2.0e3, 5.0e4), n_frequencies=200,
# )
# plt.show()

## Takeaways and Next Steps

- A pickup coil measures a derivative. Every index fitted to its spectrum is the field's plus
  two, and VAFT will not integrate for you behind an axis label.
- A spectrum is always available; a mode number is not. The number of *distinct toroidal
  positions that recorded* sets what you may claim, and the fit prints it in the title before it
  prints the answer.
- Sessions 02 and 03 taught that a refused plot is often an underived quantity. This one is the
  converse: an offered plot is not automatically an answer worth having. `available_plots`
  reports whether a call will raise, not whether it will inform.
- A transient is a change measured against a floor you also measured. The floor is not zero.
- $R^2$ is part of a fitted index. A poor fit across a band containing a coherent peak is the fit
  telling you the model is wrong there.
- Where the chain from a measured mode to a rational surface breaks, say where it broke. On
  39915 it breaks twice: no second toroidal position, and no $q = 1$ or $q = 2$ surface to break
  it at.

**Not covered here, because VAFT does not do it.** Four gaps sit directly on this session's path,
and naming them is more useful than approximating them. There is no resolver from $(m, n)$ to a
surface ([#506](https://github.com/VEST-Tokamak/vaft/issues/506)) and no rotation measurement on
either shot, so no mode tracks on a spectrogram
([#460](https://github.com/VEST-Tokamak/vaft/issues/460)). There is no IRE, sawtooth, ELM or
disruption classifier anywhere in the package — only the generic onset primitives this session
used through the timing wrappers, which is why it said *onset* and *termination* and stopped.
And there is no Alfven continuum, no tearing $\Delta'$, no island width and no growth rate; two
functions in `vaft.formula.stability` look like thresholds, and both carry docstring warnings
that their coefficients are unsourced heuristics, so neither was quoted here.

**Next**: Session 05 stops measuring what fluctuated and starts asking what *could*. It takes the
equilibrium into a linear stability calculation, and then perturbs it in three dimensions.